In [2]:
# ==========================================================
# ONE-FILE TIMESERIES for Power BI
# Output: data/data_powerbi/timeseries_data_ALL.csv
# Columns: date, year, month, total_emails, avg_sentiment,
#          unique_participants, avg_email_length, top_sender
# ==========================================================
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ---------- 0) Paths ----------
BASE = Path.cwd()                                  # SNA_Project/
DATA = BASE / "data"
IN_SENT = DATA / "SentimentalAnalysis"             # enriched_emails_{YYYY_MM}.csv
OUT_DIR = DATA / "data_powerbi"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "timeseries_data_ALL.csv"

# ---------- 1) Utilities ----------
def month_key_from_name(p: Path) -> str:
    """Return 'YYYY_MM' from filename; '' if not found."""
    m = re.search(r"(\d{4})[_-](\d{2})", p.stem)
    return f"{m.group(1)}_{m.group(2)}" if m else ""

def split_emails(raw: str):
    """Split semicolon/comma-separated recipient strings into clean lowercase emails."""
    if not isinstance(raw, str) or not raw.strip():
        return []
    raw = raw.replace(",", ";")
    out = [x.strip().lower() for x in raw.split(";")]
    return [x for x in out if x and "@" in x]

def parse_date_series(s: pd.Series) -> pd.Series:
    """Robust to dd/mm/yyyy and ISO. Returns date (no time)."""
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    miss = dt.isna()
    if miss.any():
        dt2 = pd.to_datetime(s[miss], errors="coerce", dayfirst=False)
        dt.loc[miss] = dt2
    return dt.dt.date

def pick_body_column(df: pd.DataFrame) -> str | None:
    for cand in ["clean_body", "body", "text", "message_body", "content"]:
        if cand in df.columns:
            return cand
    return None

def ensure_compound(df_enriched: pd.DataFrame, month_key: str) -> pd.DataFrame:
    """
    Ensure a 'compound' column exists; if missing, join with sentiment_scores_{YYYY_MM}.csv on message_id.
    """
    if "compound" in df_enriched.columns:
        return df_enriched

    ss_path = IN_SENT / f"sentiment_scores_{month_key}.csv"
    if ss_path.exists():
        ss = pd.read_csv(ss_path)
        cols = [c for c in ["message_id", "compound", "pos", "neu", "neg", "sentiment_label"] if c in ss.columns]
        df_enriched = df_enriched.merge(ss[cols], on="message_id", how="left")
    else:
        df_enriched["compound"] = np.nan
    return df_enriched

def unique_participants_per_day(df: pd.DataFrame) -> pd.DataFrame:
    """Distinct union of sender+recipients+cc+bcc per date."""
    for col in ["recipients", "cc", "bcc"]:
        df[col] = df[col].apply(split_emails) if col in df.columns else [[] for _ in range(len(df))]

    if "sender" in df.columns:
        s = df["sender"].fillna("").str.lower()
        s = [x if "@" in x else "" for x in s]
        df["sender_list"] = [[x] if x else [] for x in s]
    else:
        df["sender_list"] = [[] for _ in range(len(df))]

    df["addr_union"] = df[["sender_list", "recipients", "cc", "bcc"]].apply(
        lambda r: list(set(r["sender_list"] + r["recipients"] + r["cc"] + r["bcc"])), axis=1
    )
    exploded = df[["date", "addr_union"]].explode("addr_union")
    exploded = exploded.dropna(subset=["addr_union"])
    return (exploded.groupby("date")["addr_union"]
            .nunique()
            .rename("unique_participants")
            .reset_index())

def avg_email_length_per_day(df: pd.DataFrame) -> pd.DataFrame:
    """Average words per email per date. Falls back to 0 if body is missing."""
    body_col = pick_body_column(df)
    if body_col is None:
        tmp = df.groupby("date")["message_id"].size().rename("n").reset_index()
        tmp["avg_email_length"] = 0.0
        return tmp[["date", "avg_email_length"]]
    words = df[body_col].astype(str).str.split().apply(len)
    df = df.assign(email_words=words)
    return (df.groupby("date")["email_words"]
            .mean()
            .rename("avg_email_length")
            .reset_index())

def top_sender_per_day(df: pd.DataFrame) -> pd.DataFrame:
    """Top sender by emails sent per date (ties resolved by sender name)."""
    if "sender" not in df.columns:
        return pd.DataFrame({"date": [], "top_sender": []})
    counts = df.groupby(["date", "sender"]).size().rename("emails_sent").reset_index()
    counts = counts.sort_values(["date", "emails_sent", "sender"], ascending=[True, False, True])
    top1 = counts.groupby("date").head(1)[["date", "sender"]].rename(columns={"sender": "top_sender"})
    return top1

# ---------- 2) Discover input files ----------
enriched_files = sorted(IN_SENT.glob("enriched_emails_*.csv"))
if not enriched_files:
    raise FileNotFoundError(f"No files found at {IN_SENT}/enriched_emails_*.csv")

all_rows = []  # rows for output ALL

# ---------- 3) Process each month and collect rows ----------
for f in enriched_files:
    month_key = month_key_from_name(f)  # 'YYYY_MM'
    if not month_key:
        print(f"Skipping {f.name} (no YYYY_MM in name)")
        continue

    year_i, month_i = [int(x) for x in month_key.split("_")]
    print(f"Processing {f.name} …")

    df = pd.read_csv(f)

    # Normalize message_id column name if needed
    if "message_id" not in df.columns:
        for alt in ["Message-ID", "msg_id", "id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: "message_id"})
                break

    # Date column detection and parsing
    date_col = None
    for c in ["date", "Date", "sent_at", "timestamp", "datetime"]:
        if c in df.columns:
            date_col = c
            break
    if date_col is None:
        raise ValueError(f"{f.name}: no usable date column found.")

    df["date"] = parse_date_series(df[date_col])

    # Ensure we have compound sentiment
    df = ensure_compound(df, month_key)

    # Drop missing dates
    df = df.dropna(subset=["date"])

    # ---------- metrics ----------
    total = df.groupby("date")["message_id"].size().rename("total_emails").reset_index()

    if "compound" in df.columns:
        avg_s = df.groupby("date")["compound"].mean().rename("avg_sentiment").reset_index()
    else:
        tmp = df.groupby("date")["message_id"].size().reset_index()
        tmp["avg_sentiment"] = 0.0
        avg_s = tmp[["date", "avg_sentiment"]]

    uniq = unique_participants_per_day(df)
    avg_len = avg_email_length_per_day(df)
    top_s = top_sender_per_day(df)

    ts = (total.merge(avg_s, on="date", how="left")
                .merge(uniq, on="date", how="left")
                .merge(avg_len, on="date", how="left")
                .merge(top_s, on="date", how="left"))

    # attach year, month from filename (not from parsed date, to match your requirement)
    ts["year"] = year_i
    ts["month"] = month_i

    all_rows.append(ts)

# ---------- 4) Concatenate and save ONE CSV ----------
ts_all = pd.concat(all_rows, ignore_index=True)
ts_all = ts_all[["date", "year", "month", "total_emails", "avg_sentiment",
                 "unique_participants", "avg_email_length", "top_sender"]]
ts_all = ts_all.sort_values(["year", "month", "date"])

ts_all.to_csv(OUT_FILE, index=False)

print("✅ Wrote:", OUT_FILE.resolve())
print("Rows:", len(ts_all))
print(ts_all.head(10))


Processing enriched_emails_1979_12.csv …
Processing enriched_emails_1986_04.csv …
Processing enriched_emails_1986_05.csv …
Processing enriched_emails_1997_01.csv …
Processing enriched_emails_1997_03.csv …
Processing enriched_emails_1997_04.csv …
Processing enriched_emails_1997_05.csv …
Processing enriched_emails_1997_06.csv …
Processing enriched_emails_1997_07.csv …
Processing enriched_emails_1997_08.csv …
Processing enriched_emails_1997_09.csv …
Processing enriched_emails_1997_10.csv …
Processing enriched_emails_1997_11.csv …
Processing enriched_emails_1998_01.csv …
Processing enriched_emails_1998_05.csv …
Processing enriched_emails_1998_09.csv …
Processing enriched_emails_1998_10.csv …
Processing enriched_emails_1998_11.csv …
Processing enriched_emails_1998_12.csv …
Processing enriched_emails_1999_01.csv …
Processing enriched_emails_1999_02.csv …
Processing enriched_emails_1999_03.csv …
Processing enriched_emails_1999_04.csv …


/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %

Processing enriched_emails_1999_05.csv …
Processing enriched_emails_1999_06.csv …
Processing enriched_emails_1999_07.csv …
Processing enriched_emails_1999_08.csv …
Processing enriched_emails_1999_09.csv …
Processing enriched_emails_1999_10.csv …
Processing enriched_emails_1999_11.csv …
Processing enriched_emails_1999_12.csv …


/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %

Processing enriched_emails_2000_01.csv …
Processing enriched_emails_2000_02.csv …


...

/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


Processing enriched_emails_2001_10.csv …
Processing enriched_emails_2001_11.csv …
Processing enriched_emails_2001_12.csv …


/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)


Processing enriched_emails_2002_01.csv …
Processing enriched_emails_2002_02.csv …
Processing enriched_emails_2002_03.csv …
Processing enriched_emails_2002_04.csv …
Processing enriched_emails_2002_05.csv …
Processing enriched_emails_2002_06.csv …
Processing enriched_emails_2002_07.csv …
Processing enriched_emails_2002_09.csv …
Processing enriched_emails_2002_10.csv …
Processing enriched_emails_2002_12.csv …
Processing enriched_emails_2004_02.csv …
Processing enriched_emails_2005_12.csv …
Processing enriched_emails_2007_02.csv …
Processing enriched_emails_2012_11.csv …
Processing enriched_emails_2020_12.csv …
Processing enriched_emails_2024_05.csv …
Processing enriched_emails_2043_12.csv …
Processing enriched_emails_2044_01.csv …
✅ Wrote: /Users/hely/Desktop/SNA_Project/data/data_powerbi/timeseries_data_ALL.csv
Rows: 1343
         date  year  month  total_emails  avg_sentiment  unique_participants  \
0  1979-12-31  1979     12           522       0.462275                  958   
1  1986-

/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
/var/folders/_h/p75qfj7d6fz_gs7hjc6bt9980000gn/T/ipykernel_6742/788565030.py:36: UserWarning: Parsing dates in %

In [4]:
# ==========================================================
# ONE-FILE SENTIMENT SCORES for Power BI
# Output: data/data_powerbi/sentiment_scores_ALL.csv
# Schema: date, year, month, message_id, compound, pos, neu, neg, sentiment_label
# ==========================================================
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ---------- Paths ----------
BASE = Path.cwd()                                  # SNA_Project/
DATA = BASE / "data"
IN_SENT = DATA / "SentimentalAnalysis"             # sentiment_scores_{YYYY_MM}.csv, enriched_emails_{YYYY_MM}.csv
OUT_DIR = DATA / "data_powerbi"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "sentiment_scores_ALL.csv"

# ---------- Helpers ----------
def month_key_from_name(p: Path) -> str:
    m = re.search(r"(\d{4})[_-](\d{2})", p.stem)
    return f"{m.group(1)}_{m.group(2)}" if m else ""

def parse_date_series(s: pd.Series) -> pd.Series:
    # Handle ISO + timezones cleanly, no warnings
    dt = pd.to_datetime(s, errors="coerce", dayfirst=False)
    return dt.dt.date

def normalize_scores_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names if alternatives are present."""
    ren = {}
    # common variants
    for a,b in {
        "Message-ID":"message_id",
        "msg_id":"message_id",
        "id":"message_id",
        "sentiment":"compound",   # sometimes stored as 'sentiment'
        "compound_score":"compound",
        "positive":"pos", "negative":"neg", "neutral":"neu",
        "SentimentLabel":"sentiment_label",
        "label":"sentiment_label",
        "Date":"date", "timestamp":"date", "datetime":"date", "sent_at":"date"
    }.items():
        if a in df.columns and b not in df.columns:
            ren[a] = b
    if ren:
        df = df.rename(columns=ren)
    return df

def prefer(a, b):
    """Return series preferring a, fillna with b."""
    if a is None: return b
    if b is None: return a
    return a.where(a.notna(), b)

# ---------- Discover monthly inputs ----------
score_files = sorted(IN_SENT.glob("sentiment_scores_*.csv"))
enriched_files = { month_key_from_name(p): p for p in IN_SENT.glob("enriched_emails_*.csv") }

if not score_files and not enriched_files:
    raise FileNotFoundError("No sentiment_scores_*.csv or enriched_emails_*.csv found under data/SentimentalAnalysis")

all_month_rows = []

# Use union of months present in either scores or enriched
months = sorted(set([month_key_from_name(p) for p in score_files]) | set(enriched_files.keys()))

print(f"Months detected: {len(months)}")

for mkey in months:
    print(f"Processing month: {mkey}")
    year_i, month_i = [int(x) for x in mkey.split("_")]

    # Load scores (if exists)
    sc_path = IN_SENT / f"sentiment_scores_{mkey}.csv"
    df_sc = pd.read_csv(sc_path) if sc_path.exists() else pd.DataFrame()
    if not df_sc.empty:
        df_sc = normalize_scores_columns(df_sc)

    # Load enriched (for backfill/date join)
    en_path = enriched_files.get(mkey)
    df_en = pd.read_csv(en_path) if en_path and en_path.exists() else pd.DataFrame()
    if not df_en.empty:
        df_en = normalize_scores_columns(df_en)

    # Ensure message_id present from at least one source
    if "message_id" not in df_sc.columns and "message_id" not in df_en.columns:
        print(f"  ⚠️ Skipping {mkey}: no message_id in scores or enriched.")
        continue

    # If scores missing, build from enriched
    if df_sc.empty and not df_en.empty:
        df_sc = df_en[["message_id"]].copy()

    # Prepare join
    # Keep only relevant enriched cols for backfill
    keep_en = [c for c in ["message_id","date","compound","pos","neu","neg","sentiment_label"] if c in df_en.columns]
    df_en_small = df_en[keep_en].copy() if keep_en else pd.DataFrame()

    # If 'date' in either, normalize
    if "date" in df_sc.columns:
        df_sc["date"] = parse_date_series(df_sc["date"])
    if "date" in df_en_small.columns:
        df_en_small["date"] = parse_date_series(df_en_small["date"])

    # Merge (left: scores; right: enriched)
    if not df_en_small.empty:
        merged = df_sc.merge(df_en_small, on="message_id", how="left", suffixes=("", "_en"))
    else:
        merged = df_sc.copy()

    # Build final columns by preferring scores then enriched fallback
    date_col     = prefer(merged.get("date"), merged.get("date_en"))
    compound_col = prefer(merged.get("compound"), merged.get("compound_en"))
    pos_col      = prefer(merged.get("pos"), merged.get("pos_en"))
    neu_col      = prefer(merged.get("neu"), merged.get("neu_en"))
    neg_col      = prefer(merged.get("neg"), merged.get("neg_en"))
    label_col    = prefer(merged.get("sentiment_label"), merged.get("sentiment_label_en"))

    out = pd.DataFrame({
        "date": date_col,
        "message_id": merged["message_id"],
        "compound": compound_col,
        "pos": pos_col,
        "neu": neu_col,
        "neg": neg_col,
        "sentiment_label": label_col
    })

    # Add year/month from filename
    out["year"] = year_i
    out["month"] = month_i

    # Clean up: drop rows without message_id; sort
    out = out.dropna(subset=["message_id"]).reset_index(drop=True)
    out = out[["date","year","month","message_id","compound","pos","neu","neg","sentiment_label"]]
    out = out.sort_values(["year","month","date","message_id"], kind="mergesort")

    all_month_rows.append(out)

# ---------- Concatenate & write ----------
if not all_month_rows:
    raise RuntimeError("No monthly sentiment rows produced. Check input file names and columns.")

sent_all = pd.concat(all_month_rows, ignore_index=True)

# Optional: de-duplicate on (year,month,message_id) keeping first
sent_all = sent_all.drop_duplicates(subset=["year","month","message_id"], keep="first")

sent_all.to_csv(OUT_FILE, index=False)

print("✅ Wrote:", OUT_FILE.resolve())
print("Rows:", len(sent_all))
print(sent_all.head(10))


Months detected: 73
Processing month: 1979_12
Processing month: 1986_04
Processing month: 1986_05
Processing month: 1997_01
Processing month: 1997_03
Processing month: 1997_04
Processing month: 1997_05
Processing month: 1997_06
Processing month: 1997_07
Processing month: 1997_08
Processing month: 1997_09
Processing month: 1997_10
Processing month: 1997_11
Processing month: 1998_01
Processing month: 1998_05
Processing month: 1998_09
Processing month: 1998_10
Processing month: 1998_11
Processing month: 1998_12
Processing month: 1999_01
Processing month: 1999_02
Processing month: 1999_03
Processing month: 1999_04
Processing month: 1999_05
Processing month: 1999_06
Processing month: 1999_07
Processing month: 1999_08
Processing month: 1999_09
Processing month: 1999_10
Processing month: 1999_11
Processing month: 1999_12
Processing month: 2000_01
Processing month: 2000_02
Processing month: 2000_03
Processing month: 2000_04
Processing month: 2000_05
Processing month: 2000_06
Processing month: 

In [9]:
# ==========================================================
# ONE-FILE BURNOUT BAR DATA for Power BI
# Output: data/data_powerbi/burnout_bar_data_ALL.csv
# Schema: person_name, team, burnout_score, risk_level, year, month
# ==========================================================
import pandas as pd

# 1️⃣ Load your CSV
df = pd.read_csv("burnout_bar_data_ALL.csv")

# 2️⃣ Split the 'month' column into year and month parts
# Assuming the column is named 'month' — adjust if it's different
df['year'] = df['month'].astype(str).str[:4]    # First 4 chars = yyyy
df['month'] = df['month'].astype(str).str[-2:]  # Last 2 chars = mm

# 3️⃣ (Optional) Convert to int if you prefer numeric columns
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)

# 4️⃣ Save back to a new CSV
df.to_csv("burnout_bar_data_ALL_clean.csv", index=False)

# 5️⃣ Preview
print(df.head())


                      node_id  community_id  burnout_prob  burnout_label  \
0      andrew.lewis@enron.com           122      0.414012              0   
1  angela.mcculloch@enron.com            11      0.177925              0   
2         archiving@enron.com             0      0.986224              1   
3    barry.tycholiz@enron.com             0      0.173390              0   
4   benjamin.rogers@enron.com             0      0.316381              0   

   month  year  
0     12  1979  
1     12  1979  
2     12  1979  
3     12  1979  
4     12  1979  


In [1]:
# ==========================================================
# ONE-FILE NETWORK SNAPSHOT DATA for Power BI
# Output: data/data_powerbi/network_snapshot_data_ALL.csv
# Schema: node_id, x, y, pagerank, community_id, role, burnout_prob, risk_flag, year, month
# ==========================================================
import pandas as pd

# 1️⃣ Load your CSV
df = pd.read_csv("network_snapshot_data_ALL.csv")

# 2️⃣ Split the 'month' column into year and month parts
# Assuming the column is named 'month' — adjust if it's different
df['year'] = df['month'].astype(str).str[:4]    # First 4 chars = yyyy
df['month'] = df['month'].astype(str).str[-2:]  # Last 2 chars = mm

# 3️⃣ (Optional) Convert to int if you prefer numeric columns
df['year'] = df['year'].astype(int)
df['month'] = df['month'].astype(int)

# 4️⃣ Save back to a new CSV
df.to_csv("network_snapshot_data_ALLclean.csv", index=False)

# 5️⃣ Preview
print(df.head())


                 node_id         x         y  indegree  outdegree  \
0    #2.martin@enron.com  0.026961 -0.041601  0.000011        0.0   
1  '.''matthew@enron.com -0.335697 -0.182399  0.000011        0.0   
2          '.'@enron.com -0.083395  0.010836  0.000046        0.0   
3        '.'aa@enron.com -0.997707 -0.002960  0.000011        0.0   
4      '.'alan@enron.com -0.289988  0.025384  0.000011        0.0   

   betweenness  clustering_coeff  pagerank   eigenvector  closeness  \
0          0.0          0.000000  0.000008  8.584475e-07   0.039326   
1          0.0          0.000000  0.000010  1.218197e-35   0.000029   
2          0.0          0.000059  0.000010  1.427116e-06   0.042199   
3          0.0          0.000000  0.000008  1.852644e-38   0.000011   
4          0.0          0.000000  0.000008  7.697374e-08   0.041389   

   community  kcore  anomaly_score  community_id  influence_flag  \
0         12      1            NaN           NaN             NaN   
1          2      1   

In [14]:
# ==========================================================
# SINGLE FILE SNA METRICS → Power BI
# Input: data/NetworkAnalysis/sna_metrics.csv
# Output: data/data_powerbi/sna_metrics_ALL.csv
# ==========================================================
from pathlib import Path
import pandas as pd
import numpy as np

# ---------- Paths ----------
BASE = Path.cwd()                                  # SNA_Project/
DATA = BASE / "data"
IN_FILE = DATA / "NetworkAnalysis" / "sna_metrics.csv"  # ✅ Updated path
OUT_DIR = DATA / "data_powerbi"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "sna_metrics_ALL.csv"

# ---------- Load & Normalize ----------
df = pd.read_csv(IN_FILE)
print("Original columns:", df.columns.tolist())

# Rename common variants
ren = {}
for a,b in {
    "name":"node_id",
    "id":"node_id",
    "email":"node_id",
    "node":"node_id",
    "in_degree":"indegree",
    "out_degree":"outdegree",
    "betweenness_centrality":"betweenness",
    "cluster_coeff":"clustering",
    "clustering_coeff":"clustering",
    "clustering_coefficient":"clustering",
    "page_rank":"pagerank",
    "in_msgs_count":"in_msgs",
    "out_msgs_count":"out_msgs",
}.items():
    if a in df.columns and b not in df.columns:
        ren[a] = b

if ren:
    df = df.rename(columns=ren)

# Fill missing expected columns with NaN
expected_cols = ["node_id","indegree","outdegree","betweenness","clustering","pagerank","out_msgs","in_msgs"]
for col in expected_cols:
    if col not in df.columns:
        df[col] = np.nan

# Keep only required columns
df = df[expected_cols]

# Add year/month default (e.g. 0 since it's a single aggregated file)
df["year"] = 0
df["month"] = 0

# Drop rows without node_id
df = df.dropna(subset=["node_id"])

# Sort for neatness
df = df.sort_values(["node_id"])

# ---------- Save ----------
df.to_csv(OUT_FILE, index=False)

print("✅ Wrote:", OUT_FILE.resolve())
print("Rows:", len(df))
print(df.head(10))


Original columns: ['node_id', 'indegree', 'outdegree', 'betweenness', 'clustering_coeff', 'pagerank', 'eigenvector', 'closeness', 'community', 'kcore']
✅ Wrote: /Users/hely/Desktop/SNA_Project/data/data_powerbi/sna_metrics_ALL.csv
Rows: 87484
                   node_id  indegree  outdegree  betweenness  clustering  \
40     #2.martin@enron.com  0.000011        0.0          0.0         0.0   
41  #23.training@enron.com  0.000011        0.0          0.0         0.0   
42  #24.training@enron.com  0.000011        0.0          0.0         0.0   
43  #25.training@enron.com  0.000011        0.0          0.0         0.0   
44  #26.training@enron.com  0.000011        0.0          0.0         0.0   
45  #28.training@enron.com  0.000011        0.0          0.0         0.0   
46  #29.training@enron.com  0.000011        0.0          0.0         0.0   
47  #30.training@enron.com  0.000011        0.0          0.0         0.0   
48   ''blanchard@enron.com  0.000011        0.0          0.0         0.0 

In [16]:
# ==========================================================
# ONE-FILE INSIGHTS DATA for Power BI
# Output: data/data_powerbi/insights_ALL.csv
# Schema: node_id, community_id, team, total_volume, anomaly_score, anomaly_type,
#         burnout_prob, external_links, external_conn_count, year, month
# ==========================================================
from pathlib import Path
import pandas as pd
import numpy as np
import re

# ---------- Paths ----------
BASE = Path.cwd()                                  # SNA_Project/
DATA = BASE / "data"
INSIGHT_DIR = DATA / "OrganizationalInsight"
OUT_DIR = DATA / "data_powerbi"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_FILE = OUT_DIR / "insights_ALL.csv"

# ---------- Helpers ----------
def month_key_from_name(p: Path) -> str:
    m = re.search(r"(\d{4})[_-](\d{2})", p.stem)
    return f"{m.group(1)}_{m.group(2)}" if m else ""

def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    ren = {}
    for a,b in {
        "name":"node_id",
        "id":"node_id",
        "email":"node_id",
        "node":"node_id",
        "community":"community_id",
        "cluster":"community_id",
        "department":"team",
        "team":"team",
        "total_emails":"total_volume",
        "total_volume":"total_volume",
        "anomaly":"anomaly_score",
        "anomaly_score_val":"anomaly_score",
        "anomaly_label":"anomaly_type",
        "dbscan_label":"anomaly_type",
        "burnout":"burnout_prob",
        "burnout_probability":"burnout_prob",
        "external_conn":"external_conn_count",
        "external_connections":"external_conn_count"
    }.items():
        if a in df.columns and b not in df.columns:
            ren[a] = b
    if ren:
        df = df.rename(columns=ren)
    return df

# ---------- Discover insight files ----------
insight_files = sorted(INSIGHT_DIR.glob("insights_*.csv"))
if not insight_files:
    raise FileNotFoundError(f"No insights_*.csv files found in {INSIGHT_DIR}")

print(f"Found {len(insight_files)} insights files")

all_month_rows = []

# ---------- Process each file ----------
for f in insight_files:
    mkey = month_key_from_name(f)
    if not mkey:
        print(f"Skipping {f.name} (no YYYY_MM)")
        continue
    year_i, month_i = [int(x) for x in mkey.split("_")]
    print(f"Processing: {f.name}")

    df = pd.read_csv(f)
    df = normalize_cols(df)

    # Ensure node_id exists
    if "node_id" not in df.columns:
        print(f"  ⚠️ Skipping (no node_id): {f.name}")
        continue

    # Fill missing expected columns
    for col in ["community_id","team","total_volume","anomaly_score","anomaly_type","burnout_prob","external_links","external_conn_count"]:
        if col not in df.columns:
            df[col] = np.nan

    keep_cols = ["node_id","community_id","team","total_volume","anomaly_score",
                 "anomaly_type","burnout_prob","external_links","external_conn_count"]
    df = df[keep_cols]
    df["year"] = year_i
    df["month"] = month_i

    # Drop rows without node_id
    df = df.dropna(subset=["node_id"])

    all_month_rows.append(df)

# ---------- Concatenate & Save ----------
if not all_month_rows:
    raise RuntimeError("No insights rows produced — check your input files")

insights_all = pd.concat(all_month_rows, ignore_index=True)
insights_all = insights_all.drop_duplicates(subset=["node_id","year","month"], keep="first")
insights_all = insights_all.sort_values(["year","month","node_id"], kind="mergesort")

insights_all.to_csv(OUT_FILE, index=False)

print("✅ Wrote:", OUT_FILE.resolve())
print("Rows:", len(insights_all))
print(insights_all.head(10))


Found 73 insights files
Processing: insights_1979_12.csv
Processing: insights_1986_04.csv
Processing: insights_1986_05.csv
Processing: insights_1997_01.csv
Processing: insights_1997_03.csv
Processing: insights_1997_04.csv
Processing: insights_1997_05.csv
Processing: insights_1997_06.csv
Processing: insights_1997_07.csv
Processing: insights_1997_08.csv
Processing: insights_1997_09.csv
Processing: insights_1997_10.csv
Processing: insights_1997_11.csv
Processing: insights_1998_01.csv
Processing: insights_1998_05.csv
Processing: insights_1998_09.csv
Processing: insights_1998_10.csv
Processing: insights_1998_11.csv
Processing: insights_1998_12.csv
Processing: insights_1999_01.csv
Processing: insights_1999_02.csv
Processing: insights_1999_03.csv
Processing: insights_1999_04.csv
Processing: insights_1999_05.csv
Processing: insights_1999_06.csv
Processing: insights_1999_07.csv
Processing: insights_1999_08.csv
Processing: insights_1999_09.csv
Processing: insights_1999_10.csv
Processing: insight

In [18]:
# ==========================================================
# Build data/data_powerbi/insight_summary.json
# Keys: active_nodes, edges, network_density, top_influencer, anomalies_count
# ==========================================================
from pathlib import Path
import pandas as pd
import numpy as np
import json

# ---------- Paths ----------
BASE = Path.cwd()                                 # SNA_Project/
DATA = BASE / "data"
PBI  = DATA / "data_powerbi"
PBI.mkdir(parents=True, exist_ok=True)

SNAP_ALL = PBI / "network_snapshot_data_ALL.csv"  # node_id, x, y, pagerank, community_id, role, burnout_prob, risk_flag, year, month
METR_ALL = PBI / "sna_metrics_ALL.csv"            # node_id, indegree, outdegree, betweenness, clustering, pagerank, out_msgs, in_msgs, year, month
INSI_ALL = PBI / "insights_ALL.csv"               # node_id, community_id, team, total_volume, anomaly_score, anomaly_type, burnout_prob, external_links, external_conn_count, year, month

# Optional raw edges (prefer exact edge counting if present)
EDGES_DIR = DATA / "NetworkConstruction"          # network_edges_{X}.csv partitions (source,target,weight)
EDGES_GLOBS = ["network_edges_*.csv", "edges_*.csv", "*_edges_*.csv"]

# ---------- Helpers ----------
def read_if_exists(path: Path) -> pd.DataFrame | None:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return None
    return None

def find_edge_files():
    out = []
    if EDGES_DIR.exists():
        for pat in EDGES_GLOBS:
            out.extend(EDGES_DIR.glob(pat))
    # De-dup by name
    return sorted(set(out))

def count_edges_from_partitions(files):
    """
    Count undirected unique edges from partition CSVs.
    Accepts common schemas: (source,target,weight) or (src,dst,weight).
    """
    if not files:
        return None
    uniq = set()
    for f in files:
        try:
            df = pd.read_csv(f, usecols=None)   # auto-detect columns
        except Exception:
            continue
        # normalize column names
        cols = {c.lower(): c for c in df.columns}
        src = cols.get("source") or cols.get("src") or cols.get("from") or cols.get("u") or None
        dst = cols.get("target") or cols.get("dst") or cols.get("to")   or cols.get("v") or None
        if not src or not dst:
            continue
        # add undirected pair as frozenset
        for s, t in zip(df[src], df[dst]):
            if pd.isna(s) or pd.isna(t):
                continue
            uniq.add(frozenset((str(s), str(t))))
    return len(uniq)

def density_undirected(n, e):
    """Density for simple undirected graph: 2E / (N*(N-1))."""
    if n is None or e is None or n < 2:
        return None
    return (2.0 * e) / (n * (n - 1))

def density_directed(n, e):
    """Density for simple directed graph (no self-loops): E / (N*(N-1))."""
    if n is None or e is None or n < 2:
        return None
    return e / (n * (n - 1))

# ---------- Load combined sources ----------
snap = read_if_exists(SNAP_ALL)
metr = read_if_exists(METR_ALL)
insi = read_if_exists(INSI_ALL)

# ---------- active_nodes ----------
# Prefer snapshot (it’s the visualization-ready node list). Fallback to metrics if needed.
if snap is not None and "node_id" in snap.columns:
    active_nodes = snap["node_id"].nunique()
elif metr is not None and "node_id" in metr.columns:
    active_nodes = metr["node_id"].nunique()
else:
    active_nodes = 0

# ---------- edges ----------
edges = None
# 1) Try exact counting from NetworkConstruction partitions (best)
edge_files = find_edge_files()
edges = count_edges_from_partitions(edge_files)

# 2) Fallback to metrics: directed edges ≈ sum(outdegree) over a representative month (or across all)
if edges is None and metr is not None and "outdegree" in metr.columns:
    # Use overall directed edge count estimate (sum of outdegree across all nodes, dedup by (node_id, year, month) to avoid double counting)
    tmp = metr.copy()
    # If file mixes many months, pick the month with the most rows to approximate one network snapshot
    if {"year", "month"}.issubset(tmp.columns):
        grp = tmp.groupby(["year","month"]).size().sort_values(ascending=False)
        if len(grp):
            y, m = grp.index[0]
            tmp = tmp[(tmp["year"] == y) & (tmp["month"] == m)]
    edges_directed = pd.to_numeric(tmp["outdegree"], errors="coerce").fillna(0).sum()
    edges = int(edges_directed)

# ---------- network_density ----------
# Prefer undirected density if we had undirected unique E from partitions; else use directed density.
if edges is not None:
    if edge_files and edges is not None:
        network_density = density_undirected(active_nodes, edges)
    else:
        network_density = density_directed(active_nodes, edges)
else:
    network_density = None

# ---------- top_influencer ----------
# Prefer pagerank from metrics (usually cleaner); fallback to snapshot
top_influencer = None
if metr is not None and {"node_id","pagerank"}.issubset(metr.columns):
    # Take the overall max pagerank across the file
    mx = metr.loc[pd.to_numeric(metr["pagerank"], errors="coerce").fillna(-np.inf).idxmax()]
    top_influencer = str(mx["node_id"])
elif snap is not None and {"node_id","pagerank"}.issubset(snap.columns):
    mx = snap.loc[pd.to_numeric(snap["pagerank"], errors="coerce").fillna(-np.inf).idxmax()]
    top_influencer = str(mx["node_id"])

# ---------- anomalies_count ----------
anomalies_count = 0
if insi is not None:
    has_label = "anomaly_type" in insi.columns
    has_score = "anomaly_score" in insi.columns
    c = 0
    if has_label:
        c += insi["anomaly_type"].notna().sum()
    if has_score:
        # Count additional rows with numeric score > 0 (or > 95th percentile if you prefer stricter)
        scores = pd.to_numeric(insi["anomaly_score"], errors="coerce")
        c += (scores > 0).sum()
        # If both exist, this double-counts; fix by counting unique node/month pairs that satisfy either condition
        if has_label:
            mask = insi["anomaly_type"].notna() | (scores > 0)
            if {"year","month","node_id"}.issubset(insi.columns):
                c = insi.loc[mask, ["year","month","node_id"]].drop_duplicates().shape[0]
            else:
                c = int(mask.sum())
    anomalies_count = int(c)

# ---------- Build and write JSON ----------
summary = {
    "active_nodes": int(active_nodes) if active_nodes is not None else None,
    "edges": int(edges) if edges is not None else None,
    "network_density": float(network_density) if network_density is not None else None,
    "top_influencer": top_influencer,
    "anomalies_count": int(anomalies_count),
}

OUT_JSON = PBI / "insight_summary.json"
with open(OUT_JSON, "w") as f:
    json.dump(summary, f, indent=2)

print("✅ Wrote:", OUT_JSON.resolve())
print(json.dumps(summary, indent=2))


✅ Wrote: /Users/hely/Desktop/SNA_Project/data/data_powerbi/insight_summary.json
{
  "active_nodes": 50601,
  "edges": 333690,
  "network_density": 0.00026065350612124475,
  "top_influencer": "klay@enron.com",
  "anomalies_count": 48272
}
